# Hướng 3 – GNN kế thừa Knowledge Graph Hướng 2

Notebook huấn luyện R-GCN trên các quan hệ sản phẩm–thương hiệu–danh mục. Metadata từ CSV Hướng 2 được dùng làm hồ sơ sở thích yếu cho user. Không có tương tác thực nên không báo cáo Precision/Recall/NDCG.

**Cảnh báo dữ liệu:** Notebook Hướng 2 được cung cấp đang khởi tạo đồ thị rỗng và có mã giả lập user/item. Notebook này vì vậy tái dựng một đồ thị thuộc tính từ Excel, dùng `schema_metadata` trong CSV làm sở thích yếu; không coi dữ liệu đó là tương tác người dùng thật.

In [ ]:
# HƯỚNG 3 – R-GCN trên Knowledge Graph kế thừa từ Hướng 2
# Trên Google Colab: tải lên 2 file XLSX và CSV khi được yêu cầu.
from google.colab import files
import os
uploaded = files.upload()
DATA_XLSX = next((f for f in uploaded if f.lower().endswith('.xlsx')), None)
BASELINE_CSV = next((f for f in uploaded if f.lower().endswith('.csv')), None)
assert DATA_XLSX and BASELINE_CSV, 'Cần tải lên file Excel và CSV Hướng 2.'


In [ ]:
!pip -q install pandas openpyxl scikit-learn matplotlib torch

In [ ]:
import os, json, textwrap, zipfile, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
BASE='/content'
XLSX=os.path.join(BASE,DATA_XLSX)
CSV=os.path.join(BASE,BASELINE_CSV)
OUT=os.path.join(BASE,'Direction3_GNN_Output')
os.makedirs(OUT,exist_ok=True)

products=pd.read_excel(XLSX)
baseline=pd.read_csv(CSV)
required={'parent_asin','brand','main_category','sub_category','price','rating','rating_count'}
missing=required-set(products.columns)
if missing: raise ValueError(f'Missing columns in product file: {missing}')
if not {'user_id','recommended_items','score_kg','schema_metadata'}.issubset(baseline.columns):
    raise ValueError('Baseline CSV columns do not match expected schema')

# Normalize values
products=products.copy()
for c in ['parent_asin','brand','main_category','sub_category']:
    products[c]=products[c].fillna('Unknown').astype(str).str.strip()

# Create typed KG triples: Product -> Brand / Main category / Subcategory
triples=[]
for _,r in products.iterrows():
    p=r['parent_asin']
    triples += [(p,'HAS_BRAND','Brand:'+r['brand']),
                (p,'IN_CATEGORY','Category:'+r['main_category']),
                (p,'IN_SUBCATEGORY','Subcategory:'+r['sub_category'])]

# Derive user profile preference edges only from schema_metadata in Direction2 output.
# These are weak preference hints, not observed purchase/rating interactions.
user_profile_rows=[]
for _,r in baseline.iterrows():
    uid=str(r['user_id'])
    meta=str(r['schema_metadata'])
    prefs=[]
    for part in meta.split(','):
        if ':' not in part: continue
        k,v=part.split(':',1); k=k.strip().lower(); v=v.strip()
        if not v: continue
        if k=='brand':
            triples.append((uid,'PREFERS_BRAND','Brand:'+v)); prefs.append(('brand',v))
        elif k in ('cat','category','main_category'):
            triples.append((uid,'PREFERS_CATEGORY','Category:'+v)); prefs.append(('category',v))
        elif k in ('subcategory','sub_category'):
            triples.append((uid,'PREFERS_SUBCATEGORY','Subcategory:'+v)); prefs.append(('subcategory',v))
    user_profile_rows.append({'user_id':uid,'schema_metadata':meta,'parsed_preferences':'; '.join(f'{k}={v}' for k,v in prefs)})

# Add reverse relation types to propagate messages in both directions.
base_triples=list(dict.fromkeys(triples))
all_triples=[]
for h,r,t in base_triples:
    all_triples.append((h,r,t))
    all_triples.append((t,'REV_'+r,h))

entities=sorted({x for h,r,t in all_triples for x in (h,t)})
relations=sorted({r for h,r,t in all_triples})
e2i={e:i for i,e in enumerate(entities)}
r2i={r:i for i,r in enumerate(relations)}
encoded=np.array([(e2i[h],r2i[r],e2i[t]) for h,r,t in all_triples],dtype=np.int64)

# Split by original fact group so inverse edges remain in same split.
idx=np.arange(len(base_triples)); rng=np.random.default_rng(SEED); rng.shuffle(idx)
n_test=max(1,int(round(0.2*len(idx))))
test_base=set(idx[:n_test].tolist())
train_triples=[]; test_triples=[]
for j,(h,r,t) in enumerate(base_triples):
    target=test_triples if j in test_base else train_triples
    target.append((e2i[h],r2i[r],e2i[t]))
    target.append((e2i[t],r2i['REV_'+r],e2i[h]))
# Keep graph edges for message passing from train facts only.
train_arr=np.asarray(train_triples,dtype=np.int64)
test_arr=np.asarray(test_triples,dtype=np.int64)

class RelGraphConv(nn.Module):
    def __init__(self,n_entities,n_relations,dim):
        super().__init__()
        self.self_linear=nn.Linear(dim,dim,bias=False)
        self.rel_weight=nn.Parameter(torch.empty(n_relations,dim,dim))
        nn.init.xavier_uniform_(self.rel_weight)
    def forward(self,x,edge_index,edge_type):
        src,dst=edge_index
        out=self.self_linear(x)
        if len(src):
            msg=torch.bmm(x[src].unsqueeze(1),self.rel_weight[edge_type]).squeeze(1)
            agg=torch.zeros_like(x)
            agg.index_add_(0,dst,msg)
            deg=torch.zeros(x.size(0),device=x.device)
            deg.index_add_(0,dst,torch.ones_like(dst,dtype=torch.float32))
            out=out+agg/deg.clamp(min=1).unsqueeze(1)
        return F.relu(out)

class RGCNLinkPredictor(nn.Module):
    def __init__(self,n_entities,n_relations,dim=32):
        super().__init__()
        self.entity_emb=nn.Embedding(n_entities,dim)
        nn.init.xavier_uniform_(self.entity_emb.weight)
        self.conv1=RelGraphConv(n_entities,n_relations,dim)
        self.conv2=RelGraphConv(n_entities,n_relations,dim)
        self.rel_emb=nn.Embedding(n_relations,dim)
        nn.init.xavier_uniform_(self.rel_emb.weight)
    def encode(self,edge_index,edge_type):
        x=self.entity_emb.weight
        x=self.conv1(x,edge_index,edge_type)
        x=self.conv2(x,edge_index,edge_type)
        return x
    def score(self,z,trip):
        h,r,t=trip[:,0],trip[:,1],trip[:,2]
        return (z[h]*self.rel_emb(r)*z[t]).sum(dim=1)

# tensors
train_t=torch.tensor(train_arr,dtype=torch.long)
test_t=torch.tensor(test_arr,dtype=torch.long)
edge_index=train_t[:,[0,2]].T.contiguous()
edge_type=train_t[:,1].contiguous()
model=RGCNLinkPredictor(len(entities),len(relations),dim=32)
opt=torch.optim.Adam(model.parameters(),lr=0.01,weight_decay=1e-4)
all_known={(int(h),int(r),int(t)) for h,r,t in encoded.tolist()}
train_losses=[]
for epoch in range(1,201):
    model.train(); opt.zero_grad()
    z=model.encode(edge_index,edge_type)
    pos_scores=model.score(z,train_t)
    neg=train_t.clone()
    # corrupt tails; avoid exact known triples
    for i in range(len(neg)):
        h,r,_=neg[i].tolist()
        for _ in range(20):
            cand=random.randrange(len(entities))
            if (h,r,cand) not in all_known:
                neg[i,2]=cand; break
    neg_scores=model.score(z,neg)
    loss=F.softplus(-pos_scores).mean()+F.softplus(neg_scores).mean()
    loss.backward(); opt.step()
    train_losses.append(float(loss.detach()))

# Evaluate held-out KG fact ranking against 20 sampled corrupt-tail candidates.
model.eval()
with torch.no_grad():
    z=model.encode(edge_index,edge_type)
    ranks=[]
    for triple in test_t:
        h,r,t=triple.tolist()
        candidates=[t]
        for _ in range(20):
            c=random.randrange(len(entities))
            while c==t: c=random.randrange(len(entities))
            candidates.append(c)
        cand_arr=np.array([[h,r,c] for c in candidates],dtype=np.int64)
        scores=model.score(z,torch.tensor(cand_arr,dtype=torch.long)).cpu().numpy()
        rank=1+int(np.sum(scores[1:] > scores[0]))
        ranks.append(rank)
    mrr=float(np.mean([1/x for x in ranks])) if ranks else float('nan')
    hits10=float(np.mean([x<=10 for x in ranks])) if ranks else float('nan')

# GNN ranking: cosine similarity of learned user and product embeddings.
with torch.no_grad():
    z=model.encode(edge_index,edge_type).cpu().numpy()
product_ids=products['parent_asin'].tolist()
product_idx=[e2i[p] for p in product_ids]
user_ids=baseline['user_id'].astype(str).unique().tolist()
rows=[]
for uid in user_ids:
    if uid not in e2i: continue
    uvec=z[e2i[uid]].reshape(1,-1)
    pvec=z[product_idx]
    sims=cosine_similarity(uvec,pvec)[0]
    order=np.argsort(-sims)
    for rank,j in enumerate(order[:10],start=1):
        pr=products.iloc[int(j)]
        rows.append({'user_id':uid,'rank_gnn':rank,'recommended_item_gnn':pr['parent_asin'],
                     'gnn_cosine_score':float(sims[j]),'title':pr['title'],'brand':pr['brand'],
                     'main_category':pr['main_category'],'sub_category':pr['sub_category']})
gnn_recs=pd.DataFrame(rows)

# Baseline audit: are recommended product IDs present in catalog?
catalog=set(product_ids)
base_rows=[]
for _,r in baseline.iterrows():
    item=str(r['recommended_items'])
    base_rows.append({'user_id':r['user_id'],'recommended_item_h2':item,'score_kg':r['score_kg'],
                      'item_exists_in_excel_catalog':item in catalog,
                      'schema_metadata':r['schema_metadata']})
base_audit=pd.DataFrame(base_rows)

# Save outputs
pd.DataFrame(user_profile_rows).drop_duplicates().to_csv(os.path.join(OUT,'Direction3_User_Profile_From_H2.csv'),index=False)
gnn_recs.to_csv(os.path.join(OUT,'Direction3_GNN_Predictions.csv'),index=False)
base_audit.to_csv(os.path.join(OUT,'Direction2_Baseline_Audit.csv'),index=False)
metrics=pd.DataFrame([{'model':'R-GCN knowledge-graph link prediction','num_entities':len(entities),
                       'num_relations':len(relations),'num_base_triples':len(base_triples),
                       'train_triples_with_inverse':len(train_arr),'test_triples_with_inverse':len(test_arr),
                       'KG_link_prediction_MRR_sampled':mrr,'KG_link_prediction_Hits@10_sampled':hits10,
                       'epochs':200,'embedding_dim':32}])
metrics.to_csv(os.path.join(OUT,'Direction3_GNN_Metrics.csv'),index=False)

# Plot 1 training loss
plt.figure(figsize=(8,4.5)); plt.plot(range(1,len(train_losses)+1),train_losses)
plt.xlabel('Epoch'); plt.ylabel('Training loss'); plt.title('R-GCN training loss'); plt.tight_layout()
plt.savefig(os.path.join(OUT,'GNN_Training_Loss.png'),dpi=180); plt.close()
# Plot 2 top-10 average similarity by rank
if not gnn_recs.empty:
    av=gnn_recs.groupby('rank_gnn')['gnn_cosine_score'].mean()
    plt.figure(figsize=(8,4.5)); plt.plot(av.index,av.values,marker='o')
    plt.xlabel('Recommendation rank'); plt.ylabel('Mean cosine similarity'); plt.title('Mean GNN score by recommendation rank'); plt.xticks(range(1,11)); plt.tight_layout()
    plt.savefig(os.path.join(OUT,'GNN_Top10_Score.png'),dpi=180); plt.close()
# Plot 3 catalog coverage by brand
brand_counts=gnn_recs.groupby('brand')['recommended_item_gnn'].nunique().sort_values(ascending=False)
plt.figure(figsize=(8,4.5)); brand_counts.plot(kind='bar')
plt.xlabel('Brand'); plt.ylabel('Unique recommended products'); plt.title('GNN recommendation distribution by brand'); plt.tight_layout()
plt.savefig(os.path.join(OUT,'GNN_Brand_Distribution.png'),dpi=180); plt.close()

# Human-readable report
unmatched=base_audit[~base_audit['item_exists_in_excel_catalog']]
report=f'''# Hướng 3 – GNN trên Knowledge Graph kế thừa từ Hướng 2

## 1. Phạm vi và dữ liệu
- Dữ liệu sản phẩm: `{os.path.basename(XLSX)}` ({len(products)} sản phẩm).
- Dữ liệu đầu ra Hướng 2: `{os.path.basename(CSV)}` ({len(baseline)} dòng dự đoán).
- Mô hình: R-GCN (Relational Graph Convolutional Network) với bộ giải mã DistMult để học và dự đoán liên kết trong đồ thị tri thức.
- Thực thể: Product, Brand, Category, Subcategory, User.
- Quan hệ: HAS_BRAND, IN_CATEGORY, IN_SUBCATEGORY, PREFERS_BRAND, PREFERS_CATEGORY và các quan hệ đảo chiều.

## 2. Cách kế thừa Hướng 2
Trường `schema_metadata` trong CSV được phân tích để tạo các cạnh sở thích yếu của người dùng tới Brand/Category. Đây là tín hiệu hồ sơ được suy ra từ metadata của kết quả Hướng 2, không phải lịch sử mua hàng/đánh giá thực tế. Trường `score_kg` được giữ làm thông tin đối chứng, không dùng làm nhãn tương tác thật để huấn luyện GNN.

## 3. Kết quả chạy
- Số thực thể: {len(entities)}
- Số loại quan hệ: {len(relations)}
- Số bộ ba KG gốc: {len(base_triples)}
- MRR link prediction (negative sampling 20 candidates): {mrr:.4f}
- Hits@10 link prediction (negative sampling 20 candidates): {hits10:.4f}
- Số epoch: 200; embedding dimension: 32.

**Lưu ý:** MRR và Hits@10 ở đây đo khả năng khôi phục bộ ba tri thức bị giữ lại, không phải Precision@K/Recall@K/NDCG của gợi ý sản phẩm. Do không có ground-truth user-item interactions, không thể kết luận GNN tốt hơn Hướng 2 về độ chính xác gợi ý.

## 4. Giới hạn dữ liệu cần nêu trong báo cáo
Có {len(unmatched)} / {len(base_audit)} mã sản phẩm được Hướng 2 đề xuất không khớp với mã sản phẩm trong Excel. Vì vậy không thể đối chiếu trực tiếp các mã sản phẩm Hướng 2 với danh mục 30 sản phẩm. GNN sinh Top-10 trong danh mục Excel bằng cosine similarity giữa embedding User và Product đã học từ KG. Đây là kết quả thử nghiệm nguyên mẫu dựa trên đồ thị và metadata, không phải đánh giá người dùng thực tế.

## 5. Tệp kết quả
- `Direction3_GNN_Predictions.csv`: danh sách Top-10 do R-GCN xếp hạng.
- `Direction3_GNN_Metrics.csv`: chỉ số link prediction trên KG.
- `Direction2_Baseline_Audit.csv`: kiểm tra mã sản phẩm Hướng 2 có nằm trong danh mục Excel hay không.
- `Direction3_User_Profile_From_H2.csv`: hồ sơ User suy ra từ metadata.
- `GNN_Training_Loss.png`, `GNN_Top10_Score.png`, `GNN_Brand_Distribution.png`: biểu đồ.
'''
open(os.path.join(OUT,'README_Direction3_GNN.md'),'w',encoding='utf-8').write(report)

